# RSNA Knee — Training

Thin shell: pulls `knee` from GitHub at a pinned commit, trains on the pre-mounted
competition data, exports checkpoints to `/kaggle/working`.

Current experiment: **E004-dinov2-frozen** — the E003 blended-labels setup
(all 4,407 studies, soft labels, feature bank + head fits + pooled-OOF CV)
with the backbone swapped from ImageNet ResNet-34 @224 to frozen DINOv2 ViT-S/14
at its native 518px. Same labels, seed, and combiner as E003, so the CV macro is
directly comparable to E003's recorded number — that comparison is the experiment.
Requires the `WANDB_API_KEY` Kaggle secret and internet (DINOv2 weights download
at train time).

In [ ]:
# Pin a commit so every checkpoint traces to exact code. Training notebooks have internet.
# --no-deps everywhere: Kaggle's image already ships torch/timm/sklearn/numpy compiled
# together; letting pip resolve our pins upgrades numpy and breaks the whole stack.
COMMIT = "b8d29f8"  # main @ PR #14 squash: DINOv2 backbone (E004)
%pip install -q --no-deps "git+https://github.com/Josie29/capstone-rsna-knee@{COMMIT}#egg=knee"
%pip install -q --no-deps pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg pylibjpeg-rle

import numpy  # fail fast if the image stack is broken or missing
import timm
import torch

print("numpy", numpy.__version__, "| torch", torch.__version__, "| timm", timm.__version__)

from knee.series import SeriesType
from knee.train_blended import BLENDED_LABEL_SOURCE, train_blended

In [ ]:
# Competition data (DICOMs + series metadata) is pre-mounted; the blended soft
# labels arrive via the attached private knee-labels dataset (issue #2 resolved).
from pathlib import Path

from knee.data import load_blended_labels

SLUG = "rsna-knee-abnormality-detection"
# Kaggle mounts competitions under /kaggle/input/competitions/<slug> (newer layout)
# or /kaggle/input/<slug> (older docs/examples); accept either.
candidates = [Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG]
COMP_ROOT = next((p for p in candidates if (p / "train.csv").exists()), None)
if COMP_ROOT is None:
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"competition data not found; mounts: {listing}")
print("competition root:", COMP_ROOT)

# Attached datasets moved mount points too; probe the plausible locations.
LABELS_CSV = "blended_labels_v1.csv"
label_candidates = [
    Path("/kaggle/input/knee-labels") / LABELS_CSV,
    Path("/kaggle/input/datasets/josiemachalek/knee-labels") / LABELS_CSV,
    Path("/kaggle/input/datasets/knee-labels") / LABELS_CSV,
]
labels_path = next((p for p in label_candidates if p.exists()), None)
if labels_path is None:
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"{LABELS_CSV} not found; mounts: {listing}")
labels = load_blended_labels(labels_path)
print(f"blended labels: {len(labels)} studies from {labels_path}")

In [ ]:
# Metrics/hyperparams only -- no report text, no StudyInstanceUIDs (rule 2.4.b).
# Best-effort: the run proceeds without wandb if the secret isn't configured.
run = None
try:
    import wandb
    from kaggle_secrets import UserSecretsClient

    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    run = wandb.init(
        project="rsna-knee",
        config={"commit": COMMIT, "label_source": BLENDED_LABEL_SOURCE, "n_label_studies": len(labels)},
    )
except Exception as exc:  # noqa: BLE001 — telemetry must never kill a training run
    print(f"wandb disabled: {exc}")

In [ ]:
# E004: same ensemble shape and labels as E003 — only the backbone (and its native
# input size) changes, so the CV delta vs E003 is attributable to backbone +
# resolution jointly. 518px is the ViT's fixed native input (patch 14 x 37) — do not
# change one without the other. Forum context in docs/rsna_brain.md §3.5: this cheap
# A/B decides whether backbone investment continues (unfreezing next) or stops.
from knee.model import DINOV2_BACKBONE

SERIES_TYPES = [SeriesType.SAGITTAL_FLUID, SeriesType.CORONAL_FLUID, SeriesType.AXIAL_FLUID]
BACKBONE = DINOV2_BACKBONE
INPUT_SIZE = 518

CHECKPOINT_DIR = Path("/kaggle/working")
# Backbone-tagged so the E003 resnet bank and this one can never be confused.
BANK_PATH = CHECKPOINT_DIR / f"feature_bank_{BLENDED_LABEL_SOURCE}_dinov2.pt"

In [ ]:
# One threaded decode pass over all 4.4k studies fills the bank, then three head
# fits from cached features (seconds). The 518px ViT forwards make this pass
# meaningfully longer than E003's resnet@224 run — needs the T4, budget several
# hours; the persisted bank makes every later retrain/CV iteration skip it.
from knee.cv import save_feature_bank

bank, results = train_blended(
    COMP_ROOT,
    labels,
    CHECKPOINT_DIR,
    series_types=SERIES_TYPES,
    backbone=BACKBONE,
    input_size=INPUT_SIZE,
)
save_feature_bank(bank, BANK_PATH)
print("plane coverage:", {t.value: n for t, n in bank.plane_coverage().items()})
for result in results:
    print(f"{result.series_type.value}: trained on {result.n_studies} studies")
    # In-sample vs thresholded blended labels: proves the features carry signal
    # against the miner's labels; the generalization number is the CV below.
    print(result.in_sample_auc)

In [ ]:
# Local eval: pooled out-of-fold stratified CV of the full ensemble over ALL blended
# studies, same protocol/seed as E003 — same labels + seed means identical fold
# assignments, so this macro is directly comparable to E003's recorded number.
# The dinov2 checkpoints ship either way (experiments.md E004); E003's number is the
# diagnostic baseline if the LB disappoints. Caveat: labels are miner-derived, so
# this AUC measures agreement with the report miner — the LB is the truth check.
from knee.cv import cross_validate

cv = cross_validate(bank)
print(f"macro OOF AUC {cv.macro_auc:.3f} over {cv.n_repeats} repeats: "
      + ", ".join(f"{m:.3f}" for m in cv.macro_auc_per_repeat))
print({label: round(auc, 3) for label, auc in cv.per_label_auc.items()})

In [ ]:
# Checkpoints + feature bank are in /kaggle/working, which persists as notebook
# output; publish the three .pt checkpoints as a new knee-weights dataset VERSION
# that REPLACES the previous ones — the dinov2 checkpoints ship regardless of the
# CV comparison (experiments.md E004). Inference rejects duplicate series types, so
# old and new checkpoints must never be mounted together.
import math

if run is not None:
    run.config.update({"backbone": BACKBONE, "input_size": INPUT_SIZE})
    for result in results:
        wandb.log(
            {
                f"in_sample_auc/{result.series_type.value}/{label}": auc
                for label, auc in result.in_sample_auc.items()
                if not math.isnan(auc)
            }
        )
    wandb.log(
        {
            "cv/macro_auc": cv.macro_auc,
            **{f"cv/auc/{label}": auc for label, auc in cv.per_label_auc.items() if not math.isnan(auc)},
        }
    )
    run.finish()